### ingest_daily_support_tickets

In [ ]:
import sys
print(sys.executable)

In [ ]:
! pip install boto3
! pip install python-dotenv
! pip install pymysql      #Mysql driver for python to connect to sql

import os            #For env variables
import pandas as pd     # For analysis
import boto3               #Its a SDK to interact with AWS services using python
from io import StringIO    #  Treat a normal string like a file.                  #
from sqlalchemy import create_engine      #For DB connection
from datetime import datetime, timedelta   #to deal with dates

from dotenv import load_dotenv
load_dotenv("sample.env")         #To load env file



# Example: Pandas read_csv() expects a file.

# But you have CSV data as a string (API response, logs, etc.)

# Without StringIO → Pandas will throw an error.
# With StringIO → Pandas thinks it's reading a real file.

# # df (your data)
# ↓
# convert df to CSV text
# ↓
# store CSV text in memory (StringIO)
# ↓
# upload that CSV text to S3 as an object

# os whenever Python code wants to access environment variables

# PyMySQL → only helps Python talk to MySQL (low-level)
# SQLAlchemy → helps Python work with databases in a clean, powerful way (high-level)


In [ ]:
# ---------- DB CONFIG ---------- Can get from DB under schema
db_config = {
    "host": "localhost",
    "port": "3306",
    "user": "root",  # change
    "password": "root", # change
    "database": "careplus_support_db"
}

# S3 configuration(Target bucket name)
S3_BUCKET = "care-plus-project-14062026" 
S3_PREFIX = "support-tickets-db/raw_tickets/"

DATE_TRACKER_FILE = "date_tracker.txt"    # to track last processed date,open() function takes a string file name and opens the file for you.

#AWS config from env(# Access keys become env, when yOU decide to store them as environment variables)

AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),  
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION")
}

In [ ]:
# ---------- UTILITY FUNCTIONS ----------

# Create db engine
# This is a SQLAlchemy connection URL.
# It tells SQLAlchemy:
# Which database you want to connect to
# Which driver to use
# What credentials to use
# Where the database is located

def get_engine(config):
    return create_engine(f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}")

def upload_to_s3(df, bucket, key):    #df is dataframe(data),target bucket, object 
    csv_buffer = StringIO()            #data is stored in memory not in disk to improve performance     
    df.to_csv(csv_buffer, index=False)  #conveting df to csv

    s3 = boto3.client('s3', **AWS_CONFIG)    #what service to use using config,generally use ** infront of config Take the dictionary named config
                                             #and unpack it into key=value arguments.
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())    #uploading
    print(f"✅ Uploaded to s3://{bucket}/{key}")   

def read_last_date(file_path):       #last date 30 jun,read the date of july1st     
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            return f.read().strip()
    return "2025-06-30"  # Starting point before 1st July

def update_last_date(file_path, new_date):
    with open(file_path, 'w') as f:
        f.write(new_date)

def get_next_date(last_date_str):
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    next_date = last_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")

# ---------- MAIN INGESTION LOGIC ----------
def run_ingestion():
    engine = get_engine(db_config)
    last_date = read_last_date(DATE_TRACKER_FILE)
    next_date = get_next_date(last_date)

    # Query only that day’s data
    query = f"""
        SELECT * FROM support_tickets
        WHERE DATE(created_at) = '{next_date}';
    """
    df = pd.read_sql(query, engine)
    print(df.shape)
    print(df.head(10))

    if df.empty:
        print(f"⚠️ No data found for {next_date}. Skipping upload.")
        return

    # Upload to S3
    s3_key = f"{S3_PREFIX}support_tickets_{next_date}.csv"
    upload_to_s3(df, S3_BUCKET, s3_key)

    # Update date tracker
    update_last_date(DATE_TRACKER_FILE, next_date)
    print(f"📅 Updated tracker to {next_date}")

# Run
if __name__ == "__main__":
    run_ingestion()


# IMPORT happens when:
# ✔ Airflow loads your file
# ✔ Lambda loads your file
# ✔ Tests import your file
# ✔ Another script imports your file
# ✔ You split code into multiple files
# ✔ You reuse functions from this file

# Without __main__:
#     run_ingestion() runs automatically on import → BAD

# With __main__:
#     run_ingestion() runs ONLY when file is executed directly → GOOD

In [ ]:
# No need to run every time to upload every single data. So we are using looping

def run_ingestion_loop():
    while True:
        last_date = read_last_date(DATE_TRACKER_FILE)
        next_date = get_next_date(last_date)

        print(f"\n📅 Processing Date: {next_date}")

        run_ingestion()

        # Stop when no more data is available
        engine = get_engine(db_config)
        query = f"""
            SELECT * FROM support_tickets
            WHERE DATE(created_at) = '{next_date}';
        """
        df = pd.read_sql(query, engine)

        if df.empty:
            print("🚫 No more data found. Stopping pipeline.")
            break

if __name__ == "__main__":
    run_ingestion_loop()

In [ ]:
df.info()

In [ ]:
#Processed Ticket parquet file

import pandas as pd

df=pd.read_parquet("C:/Users/prath/Downloads/run-1781683450387-part-block-0-r-00000-snappy.parquet")
df.head(135)

In [ ]:

#Curated logs

import pandas as pd

df=pd.read_parquet("C:/Users/prath/Downloads/support_logs_2025-07-07_curated.parquet")
df


In [ ]:
#Curated Tickets

import pandas as pd

df=pd.read_parquet("C:/Users/prath/Downloads/part-00035-651fef90-058f-460a-88ee-746023bf10c4-c000.snappy.parquet")
df
